In [ ]:
import subprocess, sys, torch

print(sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("vram:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout[:600])

In [ ]:
import pandas as pd
import os

EVAL_BASE = "/kaggle/input/datasets/mhashamhussain/vmc2026-track2-eval/vmc2026_track2_eval_phase_distro"

meta = pd.read_csv(f"{EVAL_BASE}/metadata.csv", sep="|", header=None,
                    names=["filename", "target_emotion", "transcript"])
disk = set(os.listdir(f"{EVAL_BASE}/wav"))

avail = meta[meta.filename.isin(disk)].reset_index(drop=True)

print("metadata rows:", len(meta))
print("wavs on disk:", len(disk))
print("usable rows:", len(avail))
assert len(avail) == len(disk), "mismatch between disk files and matched metadata rows"
print(avail.head(3))

In [ ]:
!pip install git+https://github.com/sarulab-speech/UTMOSv2.git --quiet

In [ ]:
import utmosv2
print(utmosv2.__file__)
import torch
print("torch after install:", torch.__version__)
print("cuda still available:", torch.cuda.is_available())

In [ ]:
import time

t0 = time.time()
model = utmosv2.create_model(pretrained=True)
print("load time (s):", round(time.time() - t0, 1))
print(type(model))
print(next(model.parameters()).device)

In [ ]:
model = model.to("cuda")
print(next(model.parameters()).device)

In [ ]:
import librosa

sample_files = avail.filename.head(3).tolist()

for fn in sample_files:
    path = f"{EVAL_BASE}/wav/{fn}"
    y, sr = librosa.load(path, sr=16000, mono=True)
    pred = model.predict(data=y, sr=16000)
    print(fn, "->", pred, type(pred))

In [ ]:
import json
from tqdm.auto import tqdm

CKPT_PATH = "/kaggle/working/qmos_checkpoint.json"

preds = {}
failed = []

for fn in tqdm(avail.filename, total=len(avail)):
    try:
        path = f"{EVAL_BASE}/wav/{fn}"
        y, sr = librosa.load(path, sr=16000, mono=True)
        pred = model.predict(data=y, sr=16000)
        preds[fn] = float(pred[0])
    except Exception as e:
        failed.append((fn, repr(e)))

    if len(preds) % 250 == 0 and len(preds) > 0:
        with open(CKPT_PATH, "w") as f:
            json.dump(preds, f)

with open(CKPT_PATH, "w") as f:
    json.dump(preds, f)

print("succeeded:", len(preds))
print("failed:", len(failed))
print(failed[:5])

In [ ]:
import os
print(os.listdir("/kaggle/working"))

In [ ]:
print("succeeded:", len(preds))
print("failed:", len(failed))

Stop GPU here and import lib and set variables again

In [ ]:
import os, json
import pandas as pd


EVAL_BASE = "/kaggle/input/datasets/mhashamhussain/vmc2026-track2-eval/vmc2026_track2_eval_phase_distro"
meta = pd.read_csv(f"{EVAL_BASE}/metadata.csv", sep="|", header=None,
                    names=["filename", "target_emotion", "transcript"])
disk = set(os.listdir(f"{EVAL_BASE}/wav"))
avail = meta[meta.filename.isin(disk)].reset_index(drop=True)


with open("/kaggle/working/qmos_checkpoint.json") as f:
    preds = json.load(f)

print("avail rows:", len(avail))
print("preds loaded:", len(preds))
assert len(preds) == len(avail) == 2574


sub = pd.DataFrame({
    "wav": avail.filename,
    "QMOS": avail.filename.map(preds),
    "EMOS": avail.filename.map(preds),
})

assert sub.QMOS.isna().sum() == 0, "some files missing from preds"
assert sub.EMOS.isna().sum() == 0

sub.to_csv("/kaggle/working/answer.txt", index=False)
print(sub.shape)
print(sub.head(3).to_string())
print(sub.QMOS.describe())

In [ ]:
import subprocess

result = subprocess.run(
    ["zip", "-j", "/kaggle/working/submission.zip", "/kaggle/working/answer.txt"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)
print("return code:", result.returncode)

import zipfile
with zipfile.ZipFile("/kaggle/working/submission.zip") as z:
    print(z.namelist())

# Training Data (Future Work)

In [ ]:
import os
import pandas as pd

TRAIN_BASE = "/kaggle/input/datasets/mhashamhussain/vmc2026-track2-train/vmc2026-track2"

train = pd.read_csv(f"{TRAIN_BASE}/sets/train.csv", sep="|")
dev   = pd.read_csv(f"{TRAIN_BASE}/sets/dev.scp", header=None, names=["wavID"])
disk_train = set(os.listdir(f"{TRAIN_BASE}/wav"))

print("rating rows:", len(train))
print("unique wavs rated:", train.wavID.nunique())
print("unique listeners:", train.lisID.nunique())
print("dev.scp entries:", len(dev))
print("wavs on disk:", len(disk_train))
print("expected total:", train.wavID.nunique() + len(dev))
print("ratings per clip:", round(len(train) / train.wavID.nunique(), 1))
print(train.dtypes)